# 11 — V2 scientific quality assurance
**Purpose:** audit completeness, matched-period climate, feature provenance, Sentinel-2 sensitivity, and Sentinel-1 orbit/angle usability.  
**Inputs:** cached daily climate and prior phase outputs; optional Earth Engine sensor QA.  
**Outputs:** machine-readable QA tables under `outputs/qa/`.  
**Runtime:** local QA is under two minutes; full Sentinel-1 candidate QA can take hours because reductions are deliberately sequential.  
**Dependencies:** completed source notebooks, authenticated Earth Engine for optional sensor QA, and packages in `requirements.txt`.

In [6]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/langtang-glacier-ai'
%pip -q install -r requirements.txt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/langtang-glacier-ai


In [7]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import pandas as pd
from src.climate_features import (
    build_antecedent_climate_summary, matched_period_climatology,
)
from src.config import SETTINGS
from src.feature_engineering import get_feature_provenance
from src.gee_utils import build_rois, initialize_earth_engine
from src.glacier_features import (
    extract_ndsi_threshold_sensitivity, sentinel2_proxy_trend_sensitivity,
)
from src.sar_features import (
    angle_mask_sensitivity, diagnose_sentinel1_candidates,
    select_sentinel1_candidate,
)
from src.utils import ensure_output_directories
ensure_output_directories()


## Climate completeness and matched-period climatology
August 2026 accumulated metrics are not compared with complete historical months. This analysis uses 1–24 August (end-exclusive 25 August) and event-exclusive antecedent windows, each matched to identical calendar days in baseline years.

In [8]:
daily_path = SETTINGS.data_processed / 'era5_land_daily_1984_2026.csv'
daily = pd.read_csv(daily_path, parse_dates=['date'])
event_date = SETTINGS.documented_event_date
event_source = SETTINGS.documented_event_source
matched_august = matched_period_climatology(
    daily, '2026-08-01', '2026-08-25', 1984, 2025
)
antecedent = build_antecedent_climate_summary(
    daily, event_date, 1984, 2025
)
for table in (matched_august, antecedent):
    table['documented_event_date'] = event_date
    table['documented_event_source'] = event_source
matched_august.to_csv(
    SETTINGS.output_qa / 'climate_matched_august_1_24.csv', index=False
)
antecedent.to_csv(
    SETTINGS.output_qa / 'climate_event_antecedent_windows.csv', index=False
)
print('Documented event provenance:', event_date, event_source)
display(matched_august)
display(antecedent)


Documented event provenance: 2026-08-26 https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2/Nepal_flash_flood_imaged_by_satellites


,period_start,period_end_exclusive,metric,value,historical_percentile,z_score,historical_sample_size,observation_count,expected_observation_count,period_completeness,is_complete_period,quality_status,quality_reason,documented_event_date,documented_event_source
0,2026-08-01,2026-08-25,temp_mean_c,7.876645,0.952381,1.794395,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
1,2026-08-01,2026-08-25,precip_mm,357.084552,0.690476,0.332959,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
2,2026-08-01,2026-08-25,pdd,189.039476,0.952381,1.794395,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
3,2026-08-01,2026-08-25,freeze_thaw_cycles,0.000000,0.952381,-0.216430,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
4,2026-08-01,2026-08-25,snowfall_mm_we,5.263566,0.047619,-1.107048,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
5,2026-08-01,2026-08-25,runoff_mm,325.171237,0.690476,0.424993,42,24,24,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...


,window_days,period_start,period_end_exclusive,metric,value,historical_percentile,z_score,historical_sample_size,observation_count,expected_observation_count,period_completeness,is_complete_period,quality_status,quality_reason,documented_event_date,documented_event_source
0,1,2026-08-25,2026-08-26,temp_mean_c,8.023373,1.000000,1.939826,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
1,1,2026-08-25,2026-08-26,precip_mm,6.575623,0.190476,-0.802633,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
2,1,2026-08-25,2026-08-26,pdd,8.023373,1.000000,1.939826,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
3,1,2026-08-25,2026-08-26,freeze_thaw_cycles,0.000000,1.000000,NaN,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
4,1,2026-08-25,2026-08-26,snowfall_mm_we,0.019160,0.119048,-0.499208,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
5,1,2026-08-25,2026-08-26,runoff_mm,8.402011,0.261905,-0.666540,42,1,1,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
6,3,2026-08-23,2026-08-26,temp_mean_c,8.043750,1.000000,2.159280,42,3,3,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
7,3,2026-08-23,2026-08-26,precip_mm,20.795916,0.166667,-0.941256,42,3,3,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
8,3,2026-08-23,2026-08-26,pdd,24.131251,1.000000,2.159280,42,3,3,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...
9,3,2026-08-23,2026-08-26,freeze_thaw_cycles,0.000000,1.000000,NaN,42,3,3,1.0,True,GOOD,matched period is complete,2026-08-26,https://www.esa.int/Applications/Observing_the...


## Feature provenance
Every V2 engineered feature records its source, operation, and units. This table is part of the research audit trail.

In [9]:
provenance = get_feature_provenance()
provenance.to_csv(
    SETTINGS.output_qa / 'feature_provenance.csv', index=False
)
display(provenance)


,feature,source,operation,units
0,NDSI_change,Sentinel-2,monthly difference,index
1,PDD_change,ERA5-Land,monthly difference,degC day
2,antecedent_precipitation_index,ERA5-Land,API decay=0.85,mm
3,environmental_transition_magnitude,multisensor,RMS robust standardized monthly changes,relative
4,freeze_thaw_cycles,ERA5-Land,monthly sum of Tmin <= 0 < Tmax days,days
5,snow_fraction_change,Sentinel-2,monthly difference,fraction
6,temp_change,ERA5-Land,monthly difference,degC


## Optional full Earth Engine sensor QA
Set `RUN_EE_SENSOR_QA=True` to enumerate and reduce every Sentinel-1 pass/orbit candidate, compare angle masks, and run Sentinel-2 NDSI/coverage sensitivity. Candidate selection uses measured usable months, ROI coverage, and temporal span—not raw scene count. Do not interrupt the Sentinel-1 loop unless you intend to rerun it.

In [10]:
RUN_EE_SENSOR_QA = True
if RUN_EE_SENSOR_QA:
    EE_PROJECT = os.environ.get('EE_PROJECT') or input(
        'Earth Engine Cloud project ID: '
    ).strip()
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        initialize_earth_engine(project=EE_PROJECT, authenticate=True)
    rois = build_rois()

    s1_qa = diagnose_sentinel1_candidates(
        rois['glacier_roi'], event_date=event_date
    )
    selected_track = select_sentinel1_candidate(s1_qa)
    s1_qa['selected'] = (
        s1_qa['orbit_pass'].eq(selected_track['orbit_pass'])
        & s1_qa['relative_orbit'].eq(selected_track['relative_orbit'])
    )
    s1_qa.to_csv(
        SETTINGS.output_qa / 'sentinel1_candidate_track_qa.csv', index=False
    )
    angle_qa = angle_mask_sensitivity(
        rois['glacier_roi'], selected_track,
        SETTINGS.sentinel1_start, SETTINGS.sentinel1_end,
    )
    angle_qa.to_csv(
        SETTINGS.output_qa / 'sentinel1_angle_mask_sensitivity.csv',
        index=False,
    )

    s2_qa = extract_ndsi_threshold_sensitivity(rois['glacier_roi'])
    s2_qa.to_csv(
        SETTINGS.output_qa / 'sentinel2_ndsi_sensitivity_metrics.csv',
        index=False,
    )
    s2_trends = sentinel2_proxy_trend_sensitivity(s2_qa)
    s2_trends.to_csv(
        SETTINGS.output_qa / 'sentinel2_proxy_trend_sensitivity.csv',
        index=False,
    )
    display(s1_qa)
    display(selected_track)
    display(angle_qa)
    display(s2_trends)
else:
    print('Earth Engine sensor QA skipped; set RUN_EE_SENSOR_QA=True when ready.')


Earth Engine Cloud project ID: my-youtube-api-keys-453716
Sentinel-1 QA complete: ASCENDING 85
Sentinel-1 QA complete: DESCENDING 19
Sentinel-1 QA complete: DESCENDING 121


,orbit_pass,relative_orbit,total_scenes,first_acquisition,last_acquisition,temporal_coverage_days,event_date,event_source,pre_event_scenes_60d,post_event_scenes_60d,usable_months,median_valid_area_fraction,median_incidence_angle,incidence_angle_p05,incidence_angle_p95,selected
0,ASCENDING,85,289,2015-10-30,2026-08-28,3955,2026-08-26,https://www.esa.int/Applications/Observing_the...,5,1,116,0.955847,40.182658,39.908586,40.205206,True
1,DESCENDING,19,326,2016-09-26,2026-08-24,3618,2026-08-26,https://www.esa.int/Applications/Observing_the...,5,0,120,0.979943,33.452755,33.425075,33.793341,False
2,DESCENDING,121,301,2016-10-03,2026-08-31,3618,2026-08-26,https://www.esa.int/Applications/Observing_the...,5,1,9,0.000000,44.445142,44.440783,44.450090,False


{'orbit_pass': 'ASCENDING',
 'relative_orbit': 85,
 'selection_score': 0.9564135976103476,
 'selection_method': 'usable_coverage_temporal_geometry_score'}

,angle_mode,angle_min,angle_max,scene_count,valid_area_fraction,mean_angle
0,none,NaN,NaN,289,0.956865,40.181393
1,fixed,30.000000,45.000000,289,0.956865,40.181393
2,robust,40.076027,40.247925,289,0.955847,40.185370


,ndsi_threshold,coverage_threshold,n,slope_per_year,trend_direction,metric_label,direction_stable_across_coverage
0,0.3,0.0,9,0.016322,increasing,snow/clean-ice spectral proxy; not glacier area,True
1,0.3,0.6,9,0.016322,increasing,snow/clean-ice spectral proxy; not glacier area,True
2,0.3,0.8,9,0.016322,increasing,snow/clean-ice spectral proxy; not glacier area,True
3,0.3,0.9,9,0.016322,increasing,snow/clean-ice spectral proxy; not glacier area,True
4,0.4,0.0,9,0.017072,increasing,snow/clean-ice spectral proxy; not glacier area,True
5,0.4,0.6,9,0.017072,increasing,snow/clean-ice spectral proxy; not glacier area,True
6,0.4,0.8,9,0.017072,increasing,snow/clean-ice spectral proxy; not glacier area,True
7,0.4,0.9,9,0.017072,increasing,snow/clean-ice spectral proxy; not glacier area,True
8,0.5,0.0,9,0.017877,increasing,snow/clean-ice spectral proxy; not glacier area,True
9,0.5,0.6,9,0.017877,increasing,snow/clean-ice spectral proxy; not glacier area,True


## Interpretation checkpoint
A failed QA test is a result, not an inconvenience. Do not relax coverage, completeness, orbit, or geometry requirements solely to obtain a numerical comparison. NDSI snow fraction remains a snow/clean-ice spectral proxy, and Sentinel-1 GRD remains backscatter—not InSAR displacement.